In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    countDistinct,
    when,
    col,
    round,
    sum
)

In [5]:
spark=(
    SparkSession.builder
    .appName("Customer retention")
    .master("local[*]")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/13 18:20:21 WARN Utils: Your hostname, MacBook-pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.174 instead (on interface en0)
26/06/13 18:20:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 18:20:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/13 18:20:22 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/13 18:20:22 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


How many customers return and buy again

Customers with more than one order are retained customers.

In [9]:
# reading Sales fact table

sales_df=spark.read.parquet("output/gold/sales_fact")
sales_df.printSchema()
print("Total Records:")
print(sales_df.count())

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- purchase_timestamp: timestamp (nullable = true)
 |-- purchase_year: integer (nullable = true)
 |-- purchase_month: integer (nullable = true)
 |-- purchase_day: integer (nullable = true)
 |-- purchase_quarter: integer (nullable = true)
 |-- purchase_weekday: string (nullable = true)

Total Records:
117601


Orders per Customer 

We need to know how many orders each customer placed.

In [13]:
customer_oders_df=(
    sales_df
    .groupby("customer_id")
    .agg(countDistinct("order_id").alias("totals_orders")
         )
)
customer_oders_df.show(50,truncate=False)
customer_oders_df.printSchema()

+--------------------------------+-------------+
|customer_id                     |totals_orders|
+--------------------------------+-------------+
|a836f6725983cd79994904405fc070aa|1            |
|72ecfc69f7d90359d9bbf70530a340c7|1            |
|dca00fb1b6171b7131da91f9df6592ac|1            |
|6f804e6a8f98ba0d151e10eccaf1ba7b|1            |
|177fc7ae2e9f2151a7273d1040eeb7a8|1            |
|57d74bba7d8b5157d8b900559eaea3b3|1            |
|60a27a1b80babef80d8b370259139469|1            |
|6d44acbb5c4b483a53f88dc8584a4326|1            |
|174cf4e5e95b5a49bac9cee9ef6cef70|1            |
|9a5d390630499694895f2d2357a9967e|1            |
|36a1aa63bf2ebcd4911e026092700610|1            |
|1187acd131fe97cfc7cde033a4f036dd|1            |
|088935be5a63fd4166d69f633d2d1156|1            |
|80447be02d8f3f3f309c7df938a08501|1            |
|87bc0b31db47421f9c67a12145740887|1            |
|c920352ccf11e554a9981c919c5f8cad|1            |
|a3f3f06060fe4f8e89d4f2b8be6a8b2a|1            |
|0cb635404b8b41e8351

Identify Repeat Customers

In [23]:
customers_orders_df = (
    customer_oders_df 
    .withColumn("is_repeated_customer",
    when(
        col("totals_orders")>1 ,
        1
    ).otherwise(0)
    ) 
)

customers_orders_df.show(50,truncate=False)

+--------------------------------+-------------+--------------------+
|customer_id                     |totals_orders|is_repeated_customer|
+--------------------------------+-------------+--------------------+
|a836f6725983cd79994904405fc070aa|1            |0                   |
|72ecfc69f7d90359d9bbf70530a340c7|1            |0                   |
|dca00fb1b6171b7131da91f9df6592ac|1            |0                   |
|6f804e6a8f98ba0d151e10eccaf1ba7b|1            |0                   |
|177fc7ae2e9f2151a7273d1040eeb7a8|1            |0                   |
|57d74bba7d8b5157d8b900559eaea3b3|1            |0                   |
|60a27a1b80babef80d8b370259139469|1            |0                   |
|6d44acbb5c4b483a53f88dc8584a4326|1            |0                   |
|174cf4e5e95b5a49bac9cee9ef6cef70|1            |0                   |
|9a5d390630499694895f2d2357a9967e|1            |0                   |
|36a1aa63bf2ebcd4911e026092700610|1            |0                   |
|1187acd131fe97cfc7c

In [26]:
total_customers = (
    customer_orders_df.count()
)
print(
    f"Total Customers: {total_customers}"
)


Total Customers: 98665
